In [ ]:
import math
from string import ascii_lowercase, digits
import re
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import zlib

def fractal_dimension(url_path):
    tokens = url_path.split('/')
    unique_tokens = set(tokens)
    return len(unique_tokens) / len(tokens) if tokens else 0
    
def kolmogorov_complexity(url):
    compressed = zlib.compress(url.encode('utf-8'))
    return len(compressed) / len(url)
    
def detect_hexadecimal(url):
    # Regex to match hexadecimal patterns (e.g., "0x1234", "abc123")
    hex_pattern = r'\b[0-9a-fA-F]{2,}\b'
    hex_matches = re.findall(hex_pattern, url)
    return len(hex_matches)

def detect_base64(url):
    # Regex to match Base64 patterns
    base64_pattern = r'\b[A-Za-z0-9+/]{4,}={0,2}\b'
    base64_matches = re.findall(base64_pattern, url)
    return len(base64_matches)

def calculate_entropy(url):
    # Count the frequency of each character in the URL
    char_count = {}
    for char in url:
        char_count[char] = char_count.get(char, 0) + 1
    
    # Calculate the total length of the URL
    total_length = len(url)
    
    # Calculate Shannon entropy
    entropy = 0
    for count in char_count.values():
        # Probability of the character
        p_x = count / total_length
        # Shannon entropy contribution for this character
        entropy -= p_x * math.log2(p_x)
    
    return entropy

In [ ]:
df = pd.read_csv("StealthPhisher2025.csv")
LABEL = df.iloc[:,-1:].columns[0]
selected_columns = ['URL',LABEL]
df = df[selected_columns]
bDF = df[df[LABEL] == 1]
mDF = df[df[LABEL] == 0]

newDF = pd.DataFrame(columns=['URL', 'WAP_Legitimate', 'WAP_Phishing'])
valid_characters = ascii_lowercase + digits + "_-"
bRatio = {char: 0 for char in valid_characters}
mRatio = {char: 0 for char in valid_characters}
total=0
for char,count in bRatio.items():
    bRatio[char] = bDF['URL'].str.count(char).sum()
    total = total + bRatio[char]
for char, count in bRatio.items():
    bRatio[char] = bRatio[char]/total

total=0
for char,count in mRatio.items():
    mRatio[char] = mDF['URL'].str.count(char).sum()
    total = total + mRatio[char]
for char, count in mRatio.items():
    mRatio[char] = mRatio[char]/total

for index, row in df.iterrows():
    urls = row['URL']
    urls = urls.replace('.','')
    bSum=0
    bLength = 0
    mSum=0
    mLength = 0
    for char in urls:
        try:
            bSum = bSum + bRatio[char]
            bLength = bLength+1
            
            mSum = mSum + mRatio[char]
            mLength = mLength+1
        except:
            pass
    bCharRatio =bSum/bLength
    mCharRatio =mSum/mLength    
    newRow = {
        'URL': row['URL'],
        'WAPLegitimate': bCharRatio,
        'WAPPhishing': mCharRatio,
        'HexPatternCnt': detect_hexadecimal(row['URL']),
        'Base64PatternCnt': detect_base64(row['URL']),
        'ShannonEntropy': calculate_entropy(row['URL']),
        'KolmogorovComplexity': kolmogorov_complexity(row['URL']),
        'FractalDimension': fractal_dimension(row['URL'])
    }
    newDF = pd.concat([newDF, pd.DataFrame([newRow])], ignore_index=True)

newDF.to_csv('ExtendedFeatures.csv',index=False)
newDF.head(5)